## Conexão dos Notebooks

<a href="https://colab.research.google.com/github/Yuoneeh/Projeto_Redes_EstDados/blob/main/notebooks/isaque/Tarefa5_Arvore_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
from pathlib import Path

# Detecta se está no Google Colab ou em ambiente local
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_PROJETO = Path("/content/drive/MyDrive/Projeto interdisciplinar")
except (ImportError, ModuleNotFoundError):
    RAIZ_PROJETO = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]

# Configura e cria os caminhos absolutos
PASTA_DATA = RAIZ_PROJETO / "data"
PROC = PASTA_DATA / "processed"
INTERIM = PASTA_DATA / "interim"
PASTA_RAW = PASTA_DATA / "raw"
PASTA_SRC = RAIZ_PROJETO / "src"

# CRIAÇÃO DA PASTA MODELS EXIGIDA
PASTA_MODELS = RAIZ_PROJETO / "models"

for p in (INTERIM, PROC, PASTA_RAW, PASTA_SRC, PASTA_MODELS):
    p.mkdir(parents=True, exist_ok=True)

## Árvore Final

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

SEED = 42
CLASSES = ["OK", "RISCO", "FALHA"]

df = pd.read_csv(PROC / "dataset_final_treino.csv")
FEATURES = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pacotes",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]

# Nas Tarefas 3, 4 e 5:
X = df[FEATURES].fillna({"z_robusto": 0.0, "aumento_pct": 0.0, "jitter_relativo": 1.0})
y = df["rotulo"]  # <-- Previsão antecipada de 12 minutos

# 1. Definição das máscaras para as três fases temporais
treino_mask = df["bloco"] == "treino"
val_mask = df["bloco"] == "validacao"
teste_mask = df["bloco"] == "teste"

# 2. Separação explícita de Treino, Validação e Teste
X_treino, y_treino = X[treino_mask], y[treino_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_teste, y_teste = X[teste_mask], y[teste_mask]

print(f"Instâncias de Treino:    {len(X_treino)} ({len(X_treino)/len(X):.1%})")
print(f"Instâncias de Validação: {len(X_val)} ({len(X_val)/len(X):.1%})")
print(f"Instâncias de Teste:     {len(X_teste)} ({len(X_teste)/len(X):.1%})")


# 1. Carregar a árvore escolhida na Tarefa 4 e medir o TESTE TEMPORAL UMA ÚNICA VEZ
arvore_final = joblib.load(PROC / "arvore_ajustada_tarefa4.joblib")
pred_teste = arvore_final.predict(X_teste)

print("=== TESTE TEMPORAL CEGO (AVALIAÇÃO FINAL) ===")
print(f"Instâncias de Teste: {len(X_teste)}")
print(f"F1 Macro no Teste: {f1_score(y_teste, pred_teste, average='macro', labels=CLASSES, zero_division=0):.4f}")
print(f"Recall de FALHA no Teste: {recall_score(y_teste, pred_teste, labels=['FALHA'], average='macro', zero_division=0):.4f}")

print("=== 2. TESTE TEMPORAL (MEDIDO UMA VEZ) ===")
print(f"F1 Macro no Teste: {f1_score(y_teste, pred_teste, average='macro', labels=CLASSES, zero_division=0):.4f}")
print(f"Recall de FALHA no Teste: {recall_score(y_teste, pred_teste, labels=['FALHA'], average='macro', zero_division=0):.4f}")
print("\nMatriz 3x3 em contagem (Teste):")
print(pd.DataFrame(confusion_matrix(y_teste, pred_teste, labels=CLASSES),
                   index=[f"Real_{c}" for c in CLASSES], columns=[f"Pred_{c}" for c in CLASSES]))
print("\nRelatório completo no Teste:")
print(classification_report(y_teste, pred_teste, labels=CLASSES, zero_division=0))

# ==============================================================================
# 3. FLUXO QUE A ÁRVORE NÃO VIU (PROTOCOLO B - HOLD-OUT DE FLUXO INTEIRO)
# ==============================================================================
# Separamos 2 fluxos inteiros (1 curto e 1 longo) que ficam 100% fora do treino
medianas_fluxo = df.groupby("fluxo_id")["mediana"].first().sort_values()
fluxos_novos = [medianas_fluxo.index[0], medianas_fluxo.index[-1]]

treino_sem_novos = (df["bloco"] == "treino") & (~df["fluxo_id"].isin(fluxos_novos))
avaliacao_novos = df["fluxo_id"].isin(fluxos_novos)

arvore_protocolo_b = DecisionTreeClassifier(**arvore_final.get_params())
arvore_protocolo_b.fit(X[treino_sem_novos], y[treino_sem_novos])

pred_novos = arvore_protocolo_b.predict(X[avaliacao_novos])
print("\n=== 3. FLUXO QUE A ÁRVORE NÃO VIU (BASELINE PRÓPRIO DO PERÍODO A) ===")
print("Fluxos separados fora do treino:", fluxos_novos)
print(classification_report(y[avaliacao_novos], pred_novos, labels=CLASSES, zero_division=0))

## Criação do metadados.json
Nova célula para gerar JSON de documentação tecnica

In [ ]:
import json

metadados = {
    "nome_modelo": "Preditor_Degradacao_RIPE_Mesh",
    "versao_modelo": "1.0",
    "ambiente": {
        "python_version": "3.x",
        "bibliotecas": ["scikit-learn", "pandas", "numpy"]
    },
    "entradas": {
        "descricao": "Features relativas de telemetria baseadas no histórico do fluxo",
        "ordem": FEATURES,
        "tipos": ["float64", "float64", "float64", "float64", "int64", "int64", "int64", "int64"]
    },
    "calculo_features": {
        "z_robusto": "(latencia_atual - mediana_baseline) / max(1.4826 * MAD_baseline, IQR_baseline / 1.349, 1.0)",
        "aumento_pct": "(latencia_atual - mediana_baseline) / mediana_baseline * 100",
        "jitter_relativo": "jitter_atual / jitter_tipico_baseline (se jitter_tipico for 0 e atual > 0, assume 3.0)",
        "timeout_atual": "1 se RTT estiver vazio ou perda_pacotes >= 100, senão 0",
        "n5_timeout": "Soma de timeout_atual nas últimas 5 medições do mesmo fluxo_id",
        "n5_aumento80": "Soma de (aumento_pct > 80) nas últimas 5 medições do mesmo fluxo_id",
        "n5_risco": "Soma de ocorrências de RISCO nas últimas 5 medições do mesmo fluxo_id"
    },
    "tratamento_excecoes": {
        "valores_ausentes": "Preenchimento: z_robusto=0.0, aumento_pct=0.0, jitter_relativo=1.0",
        "divisao_por_zero": "Impedida pela lógica do denominador (uso de max com 1.0 no z_robusto e tratamento condicional no jitter_relativo)"
    },
    "saidas": {
        "significado": "Previsão do estado da rede para 12 minutos à frente",
        "ordem_classes": ["OK", "RISCO", "FALHA"]
    },
    "horizonte_previsao": "12 minutos à frente (shift de 3 passos de 240s)",
    "requisitos_historico": "Necessário no mínimo 1500 medições prévias no Período A para montar a baseline do fluxo, e janela contínua das últimas 5 medições para as variáveis n5",
    "regras_baseline": "Fluxos com menos de 1500 medições no Período A recebem status 'baseline_insuficiente' e são ignorados.",
    "exemplo_real": {
        "entrada_exemplo": {
            "z_robusto": 1.23,
            "aumento_pct": 0.75,
            "jitter_relativo": 3.47,
            "perda_pacotes": 0.0,
            "timeout_atual": 0,
            "n5_timeout": 0,
            "n5_aumento80": 0,
            "n5_risco": 2
        },
        "saida_esperada": "RISCO"
    }
}

caminho_metadados = PASTA_MODELS / "metadados.json"
caminho_metadados.write_text(json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")
print("Metadados salvos em:", caminho_metadados)